In [ ]:
# 挂载Google Drive并上传json文件
from google.colab import drive
drive.mount('/content/drive')

# 假设你将hotpot_train_v1.1.json上传到Google Drive某个文件夹，如MyDrive/data/
DATA_PATH = '/content/drive/MyDrive/data/hotpot_train_v1.1.json'

# HotpotQA RAG Label Generation (Colab)

本Notebook用于在Google Colab上批量处理HotpotQA数据集，生成RAG可检索性标注数据。

- 支持tqdm.notebook进度条
- 支持sentence-transformers嵌入
- 支持分批embedding和检索
- 结果保存为answer_label_hotpot.json

请先上传 hotpot_train_v1.1.json 到 Colab 根目录。

In [ ]:
# 安装依赖（如未安装）
try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    !pip install -U sentence-transformers
    from sentence_transformers import SentenceTransformer

try:
    from tqdm.notebook import tqdm
except ImportError:
    !pip install tqdm
    from tqdm.notebook import tqdm

import torch
import numpy as np
import json

In [1]:
# 配置参数
DATA_PATH = '/content/hotpot_train_v1.1.json'  # 上传到Colab根目录
OUTPUT_PATH = '/content/answer_label_hotpot.json'
EMBEDDER_NAME = 'all-MiniLM-L6-v2'
TARGET_DIM = 256
TOPK = 3
BATCH_SIZE = 1024

In [2]:
# 工具函数

def split_into_chunks(text, min_length=30, sentences_per_chunk=3, overlap_sentences=1):
    sents = text.split('。')
    sents = [s.strip() for s in sents if s.strip()]
    chunks = []
    i = 0
    while i < len(sents):
        chunk = '。'.join(sents[i:i+sentences_per_chunk])
        if len(chunk) >= min_length:
            chunks.append(chunk)
        i += sentences_per_chunk - overlap_sentences
    return chunks

def clean_and_prepare_hotpotqa(hotpot_path, min_chunk_len=30, sentences_per_chunk=3, overlap_sentences=1):
    with open(hotpot_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    samples = []
    for item in tqdm(data, desc="预处理HotpotQA样本"):
        question = item.get("question", "").strip()
        gold = item.get("answer", "").strip()
        context_list = item.get("context", [])
        context = " ".join([" ".join(paragraph[1]) for paragraph in context_list])
        chunks = split_into_chunks(context, min_length=min_chunk_len, sentences_per_chunk=sentences_per_chunk, overlap_sentences=overlap_sentences)
        if not chunks or not gold:
            continue
        samples.append({
            "question": question,
            "gold": gold,
            "chunks": chunks
        })
    return samples

In [3]:
# 嵌入器封装
class SentenceTransformerEmbedder:
    def __init__(self, model_name, target_dim=None):
        self.model = SentenceTransformer(model_name)
        self.target_dim = target_dim
    def embed_batch(self, texts):
        embs = self.model.encode(texts, show_progress_bar=False, convert_to_numpy=True, batch_size=64)
        if self.target_dim and embs.shape[1] != self.target_dim:
            from sklearn.decomposition import PCA
            pca = PCA(n_components=self.target_dim)
            embs = pca.fit_transform(embs)
        return embs

# 检索器（简单余弦相似度TopK）
def retrieve(query, chunks, chunk_embs, embedder, topk=1):
    query_emb = embedder.embed_batch([query])[0]
    scores = np.dot(chunk_embs, query_emb) / (np.linalg.norm(chunk_embs, axis=1) * np.linalg.norm(query_emb) + 1e-8)
    topk_idx = np.argsort(scores)[-topk:][::-1]
    return [chunks[i] for i in topk_idx], scores[topk_idx]

In [4]:
# 主流程（直接整体解析JSON并检查完整性）
def check_json_integrity(json_path):
    import json
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        if isinstance(data, list):
            print(f"✅ JSON文件解析成功，类型为list，共{len(data)}条数据")
            return True
        else:
            print(f"❌ JSON文件不是list类型，实际类型: {type(data)}")
            return False
    except json.JSONDecodeError as e:
        print(f"❌ JSON文件解析失败: {e}")
        return False
    except Exception as e:
        print(f"❌ 其他错误: {e}")
        return False

print("检查JSON文件完整性...")
file_ok = check_json_integrity(DATA_PATH)
if not file_ok:
    raise ValueError("JSON文件不完整或损坏，请重新上传！")

print("清洗并准备HotpotQA数据...")
samples = clean_and_prepare_hotpotqa(DATA_PATH)
print(f"有效样本数: {len(samples)}")

print("加载句向量模型...")
embedder = SentenceTransformerEmbedder(EMBEDDER_NAME, target_dim=TARGET_DIM)

# 收集所有样本的 chunks
all_chunks = []
for sample in samples:
    all_chunks.extend(sample["chunks"])

print(f"总chunks数: {len(all_chunks)}，开始计算embedding...")
all_chunk_embs = []
for i in tqdm(range(0, len(all_chunks), BATCH_SIZE), desc="Embedding chunks"):
    batch = all_chunks[i:i+BATCH_SIZE]
    batch_embs = embedder.embed_batch(batch)
    all_chunk_embs.append(batch_embs)
all_chunk_embs = np.vstack(all_chunk_embs)

清洗并准备HotpotQA数据...


FileNotFoundError: [Errno 2] No such file or directory: '/content/hotpot_train_v1.1.json'

In [ ]:
# 检索与标注
answer_label_data = []
for sample in tqdm(samples, desc="RAG检索与标注"):
    question = sample["question"]
    gold = sample["gold"]
    top_chunks, _ = retrieve(question, all_chunks, all_chunk_embs, embedder, topk=1)
    top_chunk = top_chunks[0]
    if gold.strip() == top_chunk.strip() or gold in top_chunk:
        label = "可检索"
    else:
        label = "不可检索"
    answer_label_data.append({"text": top_chunk, "label": label, "question": question, "gold": gold})

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(answer_label_data, f, ensure_ascii=False, indent=2)
print(f"已保存: {OUTPUT_PATH} (总样本数: {len(answer_label_data)})")